In [ ]:
###########################################################
# 과 제 명: [A2024_S05_2] 생활인구 분석 - 경기 부천시[31050]
# 파 일 명: S2.2.서비스인구_DATA.ipynb
# 작 성 자: 한영기
###########################################################
import os, importlib, warnings; warnings.filterwarnings(action='ignore')
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from config import config_A28       as config   ; importlib.reload(config)
from utils  import functions_utils    as fu       ; importlib.reload(fu)
from utils  import functions_log      as fl       ; importlib.reload(fl)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_lime     as flime    ; importlib.reload(flime)
# DB 접속: 사내 모듈 사용 (비공개)  # 원본: from utils  import functions_utils_db as fud      ; importlib.reload(fud)

from shapely          import wkt #함수는 문자열로 표현된 지오메트리를 파싱하여 실제 지오메트리 객체로 변환
from shapely.geometry import Polygon, mapping, shape,Point
import geopandas as gpd
#==================================================================================================
# [0-1] log 기록 정의
log = fl.Logger(config.PATH_LOG, config.LOG_FILE_NM ); log.set_log_level("info")
log.info("0-1 log 기록 정의")
#==================================================================================================
# [0-2] 실행 변수 정의
log.info("0.2.실행 변수 정의")
config_info = fu.f_get_config_info(config.config_run, config.ENV_DEV)
#==================================================================================================
# [0-3] 실행 변수
log.info("0.3.실행 변수 추가"); fu.f_memory_check()
# 전역 변수 선언 부분
# config_info["TODAY"]           = "20230630"

In [ ]:
# [분석 처리 시작]
###########################################################
# 모 듈 명: 모 듈 명 - S2.2.서비스인구_DATA.ipynb
# 파 일 명: S2.2.서비스인구_DATA.ipynb
# 작 성 자: 한영기
# 설    명: 설명
# 파일 버전: v1.0 [2024.07.22]
# 작성 일자: 2024.07.22
# 수정 일자: 2024.07.22 00:00 최초 작성
###########################################################
fl.f_log_line('[START]S2.2.서비스인구_DATA.ipynb')
# 1.실행 변수 정의
log.info("1.실행 변수 정의")
path_base:    str = "2024-06-14"
V_DATE_START: str = '2022-01-01'
V_DATE_END:   str = '2024-04-30'

path_data_out     = os.path.join(config_info["PATH_DATA_COLLECT"],path_base,"01.외부")
path_data_in      = os.path.join(config_info["PATH_DATA_COLLECT"],path_base,"02.내부")
path_data_analyze = os.path.join(config_info["PATH_DATA_ANALYZE"])

path_database = os.path.join(config_info["PATH_DATA_OUTPUT_DATEBASE"],path_base)
path_pretreat = os.path.join(config_info["PATH_DATA_OUTPUT_PRETREAT"],path_base)
path_middle   = os.path.join(config_info["PATH_DATA_OUTPUT_MIDDLE"]  ,path_base)
path_result   = os.path.join(config_info["PATH_DATA_OUTPUT_RESULT"]  ,path_base)

---

In [ ]:
import pandas as pd
import glob 

In [ ]:
###########################################################################
# [2.11][DB][e20_서비스인구_livepop] ← [d20_서비스인구_livepop]
###########################################################################
# [GG_50CELL_LIVEPOP][생활인구_셀 단위 집계]
# 1.원천 데이터 정보 : [d20_서비스인구_livepop]
# 2.타켓 데이터 정보 : [e20_서비스인구_livepop]
###########################################################################
p_years = ['2022', '2023', '2024']
for p_year in p_years:
    #==========================================================================
    log_text = ["2.1",f"[e20_서비스인구_livepop_{p_year}]"]; fu.f_line()
    log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
    # 1.출력정보 저장처리
    config_info["p_gubun"]      =  "db" # 저장 형태 [csv, xls, xlsx, db, txt, shp]
    config_info["p_table_name"] = f"e20_서비스인구_livepop_{p_year}"
    config_info["p_if_exists"]  =  "replace" #[replace][append]
    #==========================================================================
    p_sql = f"""
-- [e20_서비스인구_livepop] <- [d20_서비스인구_livepop]
with t_1 as (
    select
           ETL_YMD   as 년월일
          ,ADMI_CD   as 행정코드
          ,CTY_NM    as 시군구명
          ,ADMI_NM   as 행정동명
          ,TIME_CD   as 시간구분
          ,sum(H_M_10_CNT) as H_M_10_CNT ,sum(H_M_15_CNT) as H_M_15_CNT
          ,sum(H_M_20_CNT) as H_M_20_CNT ,sum(H_M_25_CNT) as H_M_25_CNT
          ,sum(H_M_30_CNT) as H_M_30_CNT ,sum(H_M_35_CNT) as H_M_35_CNT
          ,sum(H_M_40_CNT) as H_M_40_CNT ,sum(H_M_45_CNT) as H_M_45_CNT
          ,sum(H_M_50_CNT) as H_M_50_CNT ,sum(H_M_55_CNT) as H_M_55_CNT
          ,sum(H_M_60_CNT) as H_M_60_CNT ,sum(H_M_65_CNT) as H_M_65_CNT
          ,sum(H_M_70_CNT) as H_M_70_CNT
          ,sum(H_F_10_CNT) as H_F_10_CNT ,sum(H_F_15_CNT) as H_F_15_CNT
          ,sum(H_F_20_CNT) as H_F_20_CNT ,sum(H_F_25_CNT) as H_F_25_CNT
          ,sum(H_F_30_CNT) as H_F_30_CNT ,sum(H_F_35_CNT) as H_F_35_CNT
          ,sum(H_F_40_CNT) as H_F_40_CNT ,sum(H_F_45_CNT) as H_F_45_CNT
          ,sum(H_F_50_CNT) as H_F_50_CNT ,sum(H_F_55_CNT) as H_F_55_CNT
          ,sum(H_F_60_CNT) as H_F_60_CNT ,sum(H_F_65_CNT) as H_F_65_CNT
          ,sum(H_F_70_CNT) as H_F_70_CNT
          ,sum(W_M_10_CNT) as W_M_10_CNT ,sum(W_M_15_CNT) as W_M_15_CNT
          ,sum(W_M_20_CNT) as W_M_20_CNT ,sum(W_M_25_CNT) as W_M_25_CNT
          ,sum(W_M_30_CNT) as W_M_30_CNT ,sum(W_M_35_CNT) as W_M_35_CNT
          ,sum(W_M_40_CNT) as W_M_40_CNT ,sum(W_M_45_CNT) as W_M_45_CNT
          ,sum(W_M_50_CNT) as W_M_50_CNT ,sum(W_M_55_CNT) as W_M_55_CNT
          ,sum(W_M_60_CNT) as W_M_60_CNT ,sum(W_M_65_CNT) as W_M_65_CNT
          ,sum(W_M_70_CNT) as W_M_70_CNT
          ,sum(W_F_10_CNT) as W_F_10_CNT ,sum(W_F_15_CNT) as W_F_15_CNT
          ,sum(W_F_20_CNT) as W_F_20_CNT ,sum(W_F_25_CNT) as W_F_25_CNT
          ,sum(W_F_30_CNT) as W_F_30_CNT ,sum(W_F_35_CNT) as W_F_35_CNT
          ,sum(W_F_40_CNT) as W_F_40_CNT ,sum(W_F_45_CNT) as W_F_45_CNT
          ,sum(W_F_50_CNT) as W_F_50_CNT ,sum(W_F_55_CNT) as W_F_55_CNT
          ,sum(W_F_60_CNT) as W_F_60_CNT ,sum(W_F_65_CNT) as W_F_65_CNT
          ,sum(W_F_70_CNT) as W_F_70_CNT
          ,sum(V_M_10_CNT) as V_M_10_CNT ,sum(V_M_15_CNT) as V_M_15_CNT
          ,sum(V_M_20_CNT) as V_M_20_CNT ,sum(V_M_25_CNT) as V_M_25_CNT
          ,sum(V_M_30_CNT) as V_M_30_CNT ,sum(V_M_35_CNT) as V_M_35_CNT
          ,sum(V_M_40_CNT) as V_M_40_CNT ,sum(V_M_45_CNT) as V_M_45_CNT
          ,sum(V_M_50_CNT) as V_M_50_CNT ,sum(V_M_55_CNT) as V_M_55_CNT
          ,sum(V_M_60_CNT) as V_M_60_CNT ,sum(V_M_65_CNT) as V_M_65_CNT
          ,sum(V_M_70_CNT) as V_M_70_CNT
          ,sum(V_F_10_CNT) as V_F_10_CNT ,sum(V_F_15_CNT) as V_F_15_CNT
          ,sum(V_F_20_CNT) as V_F_20_CNT ,sum(V_F_25_CNT) as V_F_25_CNT
          ,sum(V_F_30_CNT) as V_F_30_CNT ,sum(V_F_35_CNT) as V_F_35_CNT
          ,sum(V_F_40_CNT) as V_F_40_CNT ,sum(V_F_45_CNT) as V_F_45_CNT
          ,sum(V_F_50_CNT) as V_F_50_CNT ,sum(V_F_55_CNT) as V_F_55_CNT
          ,sum(V_F_60_CNT) as V_F_60_CNT ,sum(V_F_65_CNT) as V_F_65_CNT
          ,sum(V_F_70_CNT) as V_F_70_CNT
      from d20_서비스인구_livepop_{p_year} a
     group by
           ETL_YMD
          ,ADMI_NM
          ,ADMI_CD
          ,CTY_NM
          ,TIME_CD
-- )select count(*) from t_1 b; /* [7440] */
)
    select
           년월일
          ,행정코드
          ,시군구명
          ,행정동명
          ,시간구분
          ,H_M_10_CNT ,H_M_15_CNT
          ,H_M_20_CNT ,H_M_25_CNT
          ,H_M_30_CNT ,H_M_35_CNT
          ,H_M_40_CNT ,H_M_45_CNT
          ,H_M_50_CNT ,H_M_55_CNT
          ,H_M_60_CNT ,H_M_65_CNT
          ,H_M_70_CNT
          ,H_F_10_CNT ,H_F_15_CNT
          ,H_F_20_CNT ,H_F_25_CNT
          ,H_F_30_CNT ,H_F_35_CNT
          ,H_F_40_CNT ,H_F_45_CNT
          ,H_F_50_CNT ,H_F_55_CNT
          ,H_F_60_CNT ,H_F_65_CNT
          ,H_F_70_CNT
          ,W_M_10_CNT ,W_M_15_CNT
          ,W_M_20_CNT ,W_M_25_CNT
          ,W_M_30_CNT ,W_M_35_CNT
          ,W_M_40_CNT ,W_M_45_CNT
          ,W_M_50_CNT ,W_M_55_CNT
          ,W_M_60_CNT ,W_M_65_CNT
          ,W_M_70_CNT
          ,W_F_10_CNT ,W_F_15_CNT
          ,W_F_20_CNT ,W_F_25_CNT
          ,W_F_30_CNT ,W_F_35_CNT
          ,W_F_40_CNT ,W_F_45_CNT
          ,W_F_50_CNT ,W_F_55_CNT
          ,W_F_60_CNT ,W_F_65_CNT
          ,W_F_70_CNT
          ,V_M_10_CNT ,V_M_15_CNT
          ,V_M_20_CNT ,V_M_25_CNT
          ,V_M_30_CNT ,V_M_35_CNT
          ,V_M_40_CNT ,V_M_45_CNT
          ,V_M_50_CNT ,V_M_55_CNT
          ,V_M_60_CNT ,V_M_65_CNT
          ,V_M_70_CNT
          ,V_F_10_CNT ,V_F_15_CNT
          ,V_F_20_CNT ,V_F_25_CNT
          ,V_F_30_CNT ,V_F_35_CNT
          ,V_F_40_CNT ,V_F_45_CNT
          ,V_F_50_CNT ,V_F_55_CNT
          ,V_F_60_CNT ,V_F_65_CNT
          ,V_F_70_CNT
          ,CASE DAYOFWEEK(STR_TO_DATE(년월일, '%Y%m%d'))
                WHEN 1 THEN '일'
                WHEN 2 THEN '월'
                WHEN 3 THEN '화'
                WHEN 4 THEN '수'
                WHEN 5 THEN '목'
                WHEN 6 THEN '금'
                WHEN 7 THEN '토'
            END AS 요일구분
      from t_1
    """
    #==========================================================================
    log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
    try:
        conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
        log.info(f"{log_text[0]}.2.SQL 실행결과 가져오기 ")
        df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
        log.info(f"{log_text[0]}.3.테이블 DROP")
        cur.execute(f'DROP table IF EXISTS {config_info["p_table_name"]}')
        log.info(f"{log_text[0]}.4.결과저장처리")
        result = flime.f_data_save2(p_log=log, p_data_object=df, p_config_info=config_info)

        cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 년월일 VARCHAR(8) DEFAULT NULL NULL')
        cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 행정동명 VARCHAR(20) DEFAULT NULL NULL')
        cur.execute(f'CREATE INDEX {config_info["p_table_name"]}_년월일_IDX ON {config_info["p_table_name"]} (년월일,행정동명)')

        log.info(f"{log_text[0]}.[종료]{log_text[1]} [추출건수: {len(df)}] [처리시간:{fu.f_duration_time(p_s_time)}]")
    except Exception as e:
        log.error(f'      예외가 발생했습니다.[{e}]')
    finally :
        cur.close();  conn.close()
        fu.f_config_info_del(config_info)
        df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/31 02:51:52] 2.1.[종료][e20_서비스인구_livepop_2022] [추출건수: 87600] [처리시간:1:40:01]
#[INFO   ][2024/07/31 04:35:45] 2.1.[종료][e20_서비스인구_livepop_2023] [추출건수: 87600] [처리시간:1:43:53]
#[INFO   ][2024/07/31 05:22:27] 2.1.[종료][e20_서비스인구_livepop_2024] [추출건수: 107448] [처리시간:0:46:41]

In [ ]:
###########################################################################
# [2.2][DB][e21_서비스인구_livepop] ← [e20_서비스인구_livepop]
###########################################################################
# [GG_50CELL_LIVEPOP][생활인구_셀 단위 집계]
# 1.원천 데이터 정보 : [d20_서비스인구_livepop]
# 2.타켓 데이터 정보 : [e21_서비스인구_livepop]
###########################################################################
#==========================================================================
log_text = ["2.2","[e21_서비스인구_livepop]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 1.출력정보 저장처리
config_info["p_gubun"]      = "db" # 저장 형태 [csv, xls, xlsx, db, txt, shp]
config_info["p_table_name"] = "e21_서비스인구_livepop"
config_info["p_if_exists"]  = "replace" #[replace][append]
#==========================================================================
p_sql = f"""
-- [e21_서비스인구_livepop] <- [e20_서비스인구_livepop_2022]+[e20_서비스인구_livepop_2023]+[e20_서비스인구_livepop_2024]
WITH RECURSIVE months AS (
    SELECT DATE('{V_DATE_START}') AS first_day UNION ALL                 -- '20220101'
    SELECT DATE_ADD(first_day, INTERVAL 1 MONTH) FROM months
     WHERE DATE_ADD(first_day, INTERVAL 1 MONTH) <= DATE('{V_DATE_END}') -- '20240401'
), month_day_cnt as (
    SELECT DATE_FORMAT(first_day, '%Y%m') AS 년월
          ,DAY(LAST_DAY(first_day))       AS days_in_month
    from months
-- ) select * from month_day_cnt;
), month_cnt as (
    SELECT substring(년월,1,4)  AS 년 ,count(distinct 년월) AS cnt_month
      from month_day_cnt group by substring(년월,1,4)
-- ) select * from month_cnt;
), a_1 as (
    select distinct 맵핑_T as 남여구분명 from t00_기준코드 where 구분 = '남여구분'
-- )select * from a_1;
), a_2 as (
    select distinct 맵핑_T as 연령구분명 from t00_기준코드 where 구분 = '연령대_kosis'
-- )select * from a_2;
), a_3 as (
    select 'H' as `type` union all select 'V' as `type` union all select 'W' as `type`
-- )select * from a_3;
), t_1 as (
   select * from e20_서비스인구_livepop_2022 union all
   select * from e20_서비스인구_livepop_2023 union all
   select * from e20_서비스인구_livepop_2024
-- )select count(*) from t_1; /* [42408] */
), t_2 as (
    select
           substring(년월일,1,6) as 년월
          ,행정코드
          ,시군구명
          ,행정동명
          ,요일구분
          ,시간구분
          ,round(sum(H_M_10_CNT + H_M_15_CNT),0) as h_m_10
          ,round(sum(H_M_20_CNT + H_M_25_CNT),0) as h_m_20
          ,round(sum(H_M_30_CNT + H_M_35_CNT),0) as h_m_30
          ,round(sum(H_M_40_CNT + H_M_45_CNT),0) as h_m_40
          ,round(sum(H_M_50_CNT + H_M_55_CNT),0) as h_m_50
          ,round(sum(H_M_60_CNT + H_M_65_CNT),0) as h_m_60
          ,round(sum(H_M_70_CNT),0)              as h_m_70
          ,round(sum(V_M_10_CNT + V_M_15_CNT),0) as v_m_10
          ,round(sum(V_M_20_CNT + V_M_25_CNT),0) as v_m_20
          ,round(sum(V_M_30_CNT + V_M_35_CNT),0) as v_m_30
          ,round(sum(V_M_40_CNT + V_M_45_CNT),0) as v_m_40
          ,round(sum(V_M_50_CNT + V_M_55_CNT),0) as v_m_50
          ,round(sum(V_M_60_CNT + V_M_65_CNT),0) as v_m_60
          ,round(sum(V_M_70_CNT),0)              as v_m_70
          ,round(sum(W_M_10_CNT + W_M_15_CNT),0) as w_m_10
          ,round(sum(W_M_20_CNT + W_M_25_CNT),0) as w_m_20
          ,round(sum(W_M_30_CNT + W_M_35_CNT),0) as w_m_30
          ,round(sum(W_M_40_CNT + W_M_45_CNT),0) as w_m_40
          ,round(sum(W_M_50_CNT + W_M_55_CNT),0) as w_m_50
          ,round(sum(W_M_60_CNT + W_M_65_CNT),0) as w_m_60
          ,round(sum(W_M_70_CNT),0)              as w_m_70
          ,round(sum(H_F_10_CNT + H_F_15_CNT),0) as h_w_10
          ,round(sum(H_F_20_CNT + H_F_25_CNT),0) as h_w_20
          ,round(sum(H_F_30_CNT + H_F_35_CNT),0) as h_w_30
          ,round(sum(H_F_40_CNT + H_F_45_CNT),0) as h_w_40
          ,round(sum(H_F_50_CNT + H_F_55_CNT),0) as h_w_50
          ,round(sum(H_F_60_CNT + H_F_65_CNT),0) as h_w_60
          ,round(sum(H_F_70_CNT),0)              as h_w_70
          ,round(sum(V_F_10_CNT + V_F_15_CNT),0) as v_w_10
          ,round(sum(V_F_20_CNT + V_F_25_CNT),0) as v_w_20
          ,round(sum(V_F_30_CNT + V_F_35_CNT),0) as v_w_30
          ,round(sum(V_F_40_CNT + V_F_45_CNT),0) as v_w_40
          ,round(sum(V_F_50_CNT + V_F_55_CNT),0) as v_w_50
          ,round(sum(V_F_60_CNT + V_F_65_CNT),0) as v_w_60
          ,round(sum(V_F_70_CNT),0)              as v_w_70
          ,round(sum(W_F_10_CNT + W_F_15_CNT),0) as w_w_10
          ,round(sum(W_F_20_CNT + W_F_25_CNT),0) as w_w_20
          ,round(sum(W_F_30_CNT + W_F_35_CNT),0) as w_w_30
          ,round(sum(W_F_40_CNT + W_F_45_CNT),0) as w_w_40
          ,round(sum(W_F_50_CNT + W_F_55_CNT),0) as w_w_50
          ,round(sum(W_F_60_CNT + W_F_65_CNT),0) as w_w_60
          ,round(sum(W_F_70_CNT),0)              as w_w_70
      from t_1
     group by
           substring(년월일,1,6)
          ,행정코드
          ,시군구명
          ,행정동명
          ,요일구분
          ,시간구분
-- )select count(*) from t_1; /* [42408] */
), t_3 as (
    select 년월
          ,행정코드
          ,시군구명
          ,행정동명
          ,b1.남여구분명
          ,b2.연령구분명
          ,b3.`type` as 구분
          ,요일구분
          ,시간구분
          ,case when b1.남여구분명='남자' and 연령구분명='10대이하' and b3.`type`='H' then  h_m_10
                when b1.남여구분명='남자' and 연령구분명='20대'    and b3.`type`='H' then  h_m_20
                when b1.남여구분명='남자' and 연령구분명='30대'    and b3.`type`='H' then  h_m_30
                when b1.남여구분명='남자' and 연령구분명='40대'    and b3.`type`='H' then  h_m_40
                when b1.남여구분명='남자' and 연령구분명='50대'    and b3.`type`='H' then  h_m_50
                when b1.남여구분명='남자' and 연령구분명='60대이상' and b3.`type`='H' then  h_m_60 + h_m_70
                when b1.남여구분명='여자' and 연령구분명='10대이하' and b3.`type`='H' then  h_w_10
                when b1.남여구분명='여자' and 연령구분명='20대'    and b3.`type`='H' then  h_w_20
                when b1.남여구분명='여자' and 연령구분명='30대'    and b3.`type`='H' then  h_w_30
                when b1.남여구분명='여자' and 연령구분명='40대'    and b3.`type`='H' then  h_w_40
                when b1.남여구분명='여자' and 연령구분명='50대'    and b3.`type`='H' then  h_w_50
                when b1.남여구분명='여자' and 연령구분명='60대이상' and b3.`type`='H' then  h_w_60 + h_w_70
                when b1.남여구분명='남자' and 연령구분명='10대이하' and b3.`type`='W' then  w_m_10
                when b1.남여구분명='남자' and 연령구분명='20대'    and b3.`type`='W' then  w_m_20
                when b1.남여구분명='남자' and 연령구분명='30대'    and b3.`type`='W' then  w_m_30
                when b1.남여구분명='남자' and 연령구분명='40대'    and b3.`type`='W' then  w_m_40
                when b1.남여구분명='남자' and 연령구분명='50대'    and b3.`type`='W' then  w_m_50
                when b1.남여구분명='남자' and 연령구분명='60대이상' and b3.`type`='W' then  w_m_60 + w_m_70
                when b1.남여구분명='여자' and 연령구분명='10대이하' and b3.`type`='W' then  w_w_10
                when b1.남여구분명='여자' and 연령구분명='20대'    and b3.`type`='W' then  w_w_20
                when b1.남여구분명='여자' and 연령구분명='30대'    and b3.`type`='W' then  w_w_30
                when b1.남여구분명='여자' and 연령구분명='40대'    and b3.`type`='W' then  w_w_40
                when b1.남여구분명='여자' and 연령구분명='50대'    and b3.`type`='W' then  w_w_50
                when b1.남여구분명='여자' and 연령구분명='60대이상' and b3.`type`='W' then  w_w_60 + w_w_70
                when b1.남여구분명='남자' and 연령구분명='10대이하' and b3.`type`='V' then  v_m_10
                when b1.남여구분명='남자' and 연령구분명='20대'    and b3.`type`='V' then  v_m_20
                when b1.남여구분명='남자' and 연령구분명='30대'    and b3.`type`='V' then  v_m_30
                when b1.남여구분명='남자' and 연령구분명='40대'    and b3.`type`='V' then  v_m_40
                when b1.남여구분명='남자' and 연령구분명='50대'    and b3.`type`='V' then  v_m_50
                when b1.남여구분명='남자' and 연령구분명='60대이상' and b3.`type`='V' then  v_m_60 + v_m_70
                when b1.남여구분명='여자' and 연령구분명='10대이하' and b3.`type`='V' then  v_w_10
                when b1.남여구분명='여자' and 연령구분명='20대'    and b3.`type`='V' then  v_w_20
                when b1.남여구분명='여자' and 연령구분명='30대'    and b3.`type`='V' then  v_w_30
                when b1.남여구분명='여자' and 연령구분명='40대'    and b3.`type`='V' then  v_w_40
                when b1.남여구분명='여자' and 연령구분명='50대'    and b3.`type`='V' then  v_w_50
                when b1.남여구분명='여자' and 연령구분명='60대이상' and b3.`type`='V' then  v_w_60 + v_w_70
                else 0
            end as CNT
      from t_2 a
     inner join a_1 b1
     inner join a_2 b2
     inner join a_3 b3
)   /* CNT_day|CNT_month 항목생성 */
    select a.년월
          ,case when substring(a.년월,1,4)='2024' then '부천시'    else a.시군구명 end as 시군구명
          ,case when substring(a.년월,1,4)='2024' then d.구행정동명 else a.행정동명 end as 행정동명
          ,a.남여구분명
          ,a.연령구분명
          ,a.구분
          ,a.요일구분
          ,a.시간구분
          ,a.CNT
          ,round(a.CNT / b.days_in_month,1) as CNT_day
          ,round(a.CNT / c.cnt_month,1)       as CNT_month
          ,a.시군구명 as 시군구명_org
          ,a.행정동명 as 행정동명_org
      from t_3 a
      left join month_day_cnt b
        on a.년월 = b.년월
      left join month_cnt     c
        on substring(a.년월,1,4) = c.년
      left join c02_행정구역_개편_행정동코드 d
        on a.행정코드 = substring(d.행정동코드,1,8)
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    log.info(f"{log_text[0]}.2.SQL 실행결과 가져오기 ")
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f"{log_text[0]}.3.테이블 DROP")
    cur.execute(f'DROP table IF EXISTS {config_info["p_table_name"]}')
    log.info(f"{log_text[0]}.4.결과저장처리")
    result = flime.f_data_save2(p_log=log, p_data_object=df, p_config_info=config_info)

    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 년월 VARCHAR(6) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 시군구명 VARCHAR(20) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 행정동명 VARCHAR(20) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 남여구분명 VARCHAR(2) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 연령구분명 VARCHAR(20) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 구분 VARCHAR(2) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 요일구분 VARCHAR(2) DEFAULT NULL NULL')
    cur.execute(f'ALTER TABLE {config_info["p_table_name"]} MODIFY COLUMN 시간구분 VARCHAR(2) DEFAULT NULL NULL')
    cur.execute(f'CREATE INDEX {config_info["p_table_name"]}_시군구명_IDX ON {config_info["p_table_name"]} (시군구명,행정동명,남여구분명,연령구분명,구분,요일구분,시간구분)')
    cur.execute(f'CREATE INDEX {config_info["p_table_name"]}_년월_IDX ON {config_info["p_table_name"]} (년월,행정동명)')

    log.info(f"{log_text[0]}.[종료]{log_text[1]} [추출건수: {len(df)}] [처리시간:{fu.f_duration_time(p_s_time)}]")
except Exception as e:
    log.error(f'      file 예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/31 05:26:48] 2.2.[종료][e21_서비스인구_livepop] [추출건수: 2353824] [처리시간:0:04:20]

In [ ]:
###########################################################################
# [2.21][xlsx][2.서비스인구][SX_AGE_PTYPE] ← [e21_서비스인구_livepop]
###########################################################################
# 서비스인구 집계
# 1.원천 데이터 정보 : [e21_서비스인구_livepop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.21","[2.서비스인구][SX_AGE_PTYPE]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "2.서비스인구"
config_info["p_sheet_name"] = "SX_AGE_PTYPE"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [21.서비스인구][SX_AGE_PTYPE]<-[e21_서비스인구_livepop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          ,a.구분
          -- ,a.요일구분
          -- ,a.시간구분
          ,sum(a.CNT/24)       as CNT    -- 시간구분 때문에 24로 나누어줌
          ,sum(a.CNT_day/24)   as CNT_day
          ,sum(a.CNT_month/24) as CNT_month
          ,0                as 인구수
--           ,시군구명_org
--           ,행정동명_org
      from e21_서비스인구_livepop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          ,a.구분
          -- ,a.요일구분
          -- ,a.시간구분
--           ,시군구명_org
--           ,행정동명_org
-- )select count(*) from t_1; /* [2052] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          ,a.남여구분명
          ,a.연령구분명
          -- ,a.요일구분
          -- ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
          ,c.인구수
          ,c.인구수_전년
          ,a.구분
--           ,a.시군구명_org
--           ,a.행정동명_org
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       and a.남여구분명     = b.남여구분명
       and a.연령구분명     = b.연령구분명
       and a.구분         = b.구분
       -- and a.요일구분  = b.요일구분
       -- and a.시간구분  = b.시간구분
--        and a.시군구명_org  = b.시군구명_org
--        and a.행정동명_org  = b.행정동명_org
      left join e03_인구_읍면동_남여_2023기준 c
        on substring(a.년월,1,4) = c.년
       and a.행정동명  = c.행정동명
       and a.남여구분명 = c.남여구분명
       and a.연령구분명 = c.연령구분명
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/31 11:19:29] 2.21.[종료][2.서비스인구][SX_AGE_PTYPE] [저장건수: 10080,(2.서비스인구 | SX_AGE_PTYPE)] [처리시간:0:00:25]

In [ ]:
###########################################################################
# [2.22][xlsx][2.서비스인구][DOW_PTYPE] ← [e21_서비스인구_livepop]
###########################################################################
# 서비스인구 집계
# 1.원천 데이터 정보 : [e21_서비스인구_livepop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.22","[2.서비스인구][DOW_PTYPE]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "2.서비스인구"
config_info["p_sheet_name"] = "DOW_PTYPE"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [22.서비스인구][DOW_PTYPE]<-[e21_서비스인구_livepop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.구분
          ,a.요일구분
          -- ,a.시간구분
          ,sum(a.CNT/24)       as CNT
          ,sum(a.CNT_day/24)   as CNT_day
          ,sum(a.CNT_month/24) as CNT_month
          ,0                as 인구수
--           ,시군구명_org
--           ,행정동명_org
      from e21_서비스인구_livepop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.구분
          ,a.요일구분
          -- ,a.시간구분
--           ,시군구명_org
--           ,행정동명_org
-- )select count(*) from t_1; /* [2052] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.요일구분
          -- ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
          ,a.인구수
          ,b.인구수 as 인구수_전년
          ,a.구분
--           ,a.시군구명_org
--           ,a.행정동명_org
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       -- and a.남여구분명 = b.남여구분명
       -- and a.연령구분명 = b.연령구분명
       and a.구분     = b.구분
       and a.요일구분  = b.요일구분
       -- and a.시간구분  = b.시간구분
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/31 11:19:53] 2.22.[종료][2.서비스인구][DOW_PTYPE] [저장건수: 5880,(2.서비스인구 | DOW_PTYPE)] [처리시간:0:00:23]

In [ ]:
###########################################################################
# [2.23][xlsx][2.서비스인구][TIME_PTYPE] ← [e21_서비스인구_livepop]
###########################################################################
# 서비스인구 집계
# 1.원천 데이터 정보 : [e21_서비스인구_livepop]
# 2.타켓 데이터 정보 : [xlsx]
###########################################################################
# 1.로그 변수 선언
log_text = ["2.23","[2.서비스인구][TIME_PTYPE]"]; fu.f_line()
log.info(f"{log_text[0]}.[시작]{log_text[1]}"); p_s_time = fu.f_start_time()
# 2.파일 변수 선언
gubun                       = "xlsx" # 저장 형태 [csv, xlsx, db, txt, shp]
config_info["p_file_name"]  = "2.서비스인구"
config_info["p_sheet_name"] = "TIME_PTYPE"
config_info["p_file_path"]  = os.path.join(path_result,config_info["p_file_name"]); fu.mkdir_if_not_exists(config_info["p_file_path"])
# 3.SQL 데이터 추출 SQL
p_sql = f"""
-- [23.서비스인구][TIME_PTYPE]<-[e21_서비스인구_livepop]
with t_1 as (
    select a.년월
          ,substring(a.년월,1,4) as 년
          ,substring(a.년월,5,2) as 월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.구분
          -- ,a.요일구분
          ,a.시간구분
          ,sum(a.CNT)       as CNT
          ,sum(a.CNT_day)   as CNT_day
          ,sum(a.CNT_month) as CNT_month
          ,0                as 인구수
--           ,시군구명_org
--           ,행정동명_org
      from e21_서비스인구_livepop a
     group by
           a.년월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          ,a.구분
          -- ,a.요일구분
          ,a.시간구분
--           ,시군구명_org
--           ,행정동명_org
-- )select count(*) from t_1; /* [2052] */
)
    select a.년월
          ,a.년
          ,a.월
          ,a.시군구명
          ,a.행정동명
          -- ,a.남여구분명
          -- ,a.연령구분명
          -- ,a.요일구분
          ,a.시간구분
          ,round(a.CNT,1)       as CNT
          ,round(b.CNT,1)       as CNT_전년동월
          ,round(a.CNT_day,1)   as CNT_day
          ,round(b.CNT_day,1)   as CNT_day_전년동월
          ,round(a.CNT_month,1) as CNT_month
          ,round(b.CNT_month,1) as CNT_month_전년동월
          ,a.인구수
          ,b.인구수 as 인구수_전년
          ,a.구분
--           ,a.시군구명_org
--           ,a.행정동명_org
      from      t_1 a
      left join t_1 b
        on DATE_FORMAT(DATE_SUB(STR_TO_DATE(CONCAT(a.년월,'01'), '%Y%m%d'), INTERVAL 12 MONTH), '%Y%m') = b.년월
       and a.시군구명  = b.시군구명
       and a.행정동명  = b.행정동명
       -- and a.남여구분명 = b.남여구분명
       -- and a.연령구분명 = b.연령구분명
       and a.구분     = b.구분
       -- and a.요일구분  = b.요일구분
       and a.시간구분  = b.시간구분
"""
#==========================================================================
log.info(f"{log_text[0]}.1.{config_info['db_type']} 접속")
try:
    # 4.1.Database 접속
    conn = None; cur = None  # DB 접속: 사내 모듈 사용 (비공개)  # 원본: conn = flime.f_get_conn(...); cur = conn.cursor()
    df = fud.f_sql_result(p_cursor=cur,p_sql=p_sql)
    log.info(f'{log_text[0]}.2.SQL 실행결과 가져오기 [처리시간:{fu.f_duration_time(p_s_time)}] ')

    # 4.2.df 내의 열을 숫자형으로 변환
    list_col = ['CNT','CNT_전년동월','CNT_day','CNT_day_전년동월','CNT_month','CNT_month_전년동월']
    df = fu.convert_columns_to_numeric(df, list_col)

    log.info(f'{log_text[0]}.3.Excel 파일 저장하기 ')
    result = flime.f_data_to_excel(p_log = log, p_data_object = df, p_config_info = config_info)

    log.info(f'{log_text[0]}.[종료]{log_text[1]} [저장건수: {len(df)},({config_info["p_file_name"]} | {config_info["p_sheet_name"]})] [처리시간:{fu.f_duration_time(p_s_time)}]')
except Exception as e:
    log.error(f'      예외가 발생했습니다.[{e}]')
finally :
    cur.close();  conn.close()
    fu.f_config_info_del(config_info)
    df = None; fu.f_del_objects([df])
#[INFO   ][2024/07/31 11:20:18] 2.23.[종료][2.서비스인구][TIME_PTYPE] [저장건수: 20460,(2.서비스인구 | TIME_PTYPE)] [처리시간:0:00:24]